In [1]:
!pip install torch transformers transformers accelerate bitsandbytes langchain sentence-transformers faiss-cpu openpyxl datasets ragatouille gradio -q

In [2]:
!pip install gradio --upgrade # Upgrade gradio to the latest version -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 50.2 MB/s  0:00:00
  Attempting uninstall: gradio
    Found existing installation: gradio 6.31.0
    Uninstalling gradio-6.31.0:
      Successfully uninstalled gradio-6.31.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
data_path="/content/drive/MyDrive/TV GUIDE BOT/20240422 EPG.csv"

In [5]:
pip install -U langchain-core langchain-text-splitters langchain-community langchain-huggingface sentence-transformers faiss-cpu transformers gradio pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 60.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.3 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: pandas
 

In [6]:
import gradio as gr
import pandas as pd
from tqdm.notebook import tqdm
from langchain_core.documents import Document as LangchainDocument      # moved
from langchain_text_splitters import RecursiveCharacterTextSplitter     # Chunking
from transformers import AutoTokenizer                                  #
from langchain_community.vectorstores import FAISS                      # unchanged
from langchain_huggingface import HuggingFaceEmbeddings                 # moved
from langchain_community.vectorstores.utils import DistanceStrategy     # unchanged

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/tmp/ipykernel_465/2712024962.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS                      # unchanged


In [7]:
!pip install -U langchain-community -q

In [8]:
# Load your data
data_path = "/content/drive/MyDrive/TV GUIDE BOT/20240422 EPG.csv"
df = pd.read_csv(data_path, parse_dates=['event_date'])
df = df[df.deleted_at.isna()]
df = df.drop_duplicates(['duration', 'event_date', 'EPG_Program_Title', 'EPG_LogLine', 'EPG_LongSynopsis'])

In [9]:
# Drop radio channels
radio_filters = set(['FM', 'LM', 'Radio'])
df['Is_Radio'] = df.EPG_Program_Title.apply(lambda x: bool(set(x.split(' ')) & radio_filters))
df = df[df.Is_Radio == False]

df['EPG_LogLine'] = df['EPG_LogLine'].apply(lambda x: 'N/A' if x == "Press 'i'" else x)

In [10]:
# Prepare data for LangChain
def df_to_text(row):
    channel_id = f'Channel ID: {row["channel_id"]}'
    program_date = f'Program Date: {row["event_date"].date()}'
    program_time = f'Program Time: {str(row["event_date"].time())[:-3]}'  # remove the seconds, i.e 23:15:00 -> 23:15
    program_name = f'Program Name: {row["EPG_Program_Title"]}'
    program_category = f'Program Category: {row["EPG_LogLine"]}'
    program_synopsis = f'Program Synopsis: {row["EPG_LongSynopsis"]}'
    return  f"""{channel_id}\n{program_name}\n{program_category}\n{program_synopsis}"""


In [11]:
df_subset = df[['channel_id', 'duration', 'event_date', 'EPG_Program_Title', 'EPG_LogLine', 'EPG_LongSynopsis']].copy().reset_index(drop=True)
df_subset['page_content'] = df_subset.apply(df_to_text, axis=1)

In [12]:
RAW_KNOWLEDGE_BASE = [
    LangchainDocument(page_content=doc['page_content'], metadata={
        "channel_id": doc["channel_id"],
        "event_date": doc["event_date"],
        "title": doc["EPG_Program_Title"],
        "category": doc["EPG_LogLine"],
        "synopsis": doc["EPG_LongSynopsis"]
    }) for _, doc in tqdm(df_subset.iterrows())
]

0it [00:00, ?it/s]

In [13]:
# Define function to chunk documents
EMBEDDING_MODEL = "thenlper/gte-large"
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [14]:
def chunk_documents(chunk_size, knowledge_base, tokenizer):
    DOCUMENT_SEPARATORS = ["\n", "."]
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.1),
        add_start_index=True,
        strip_whitespace=True,
        separators=DOCUMENT_SEPARATORS
    )
    # The for loop should be at the same indentation level as the previous line
    docs_processed = []
    for doc in knowledge_base:
        docs_processed += text_splitter.split_documents([doc])

    unique_texts = {}
    docs_processed_unique = []
    for doc in docs_processed:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            docs_processed_unique.append(doc)

    return docs_processed_unique

In [15]:
docs_processed = chunk_documents(
    chunk_size=512,
    knowledge_base=RAW_KNOWLEDGE_BASE,
    tokenizer=tokenizer
)

In [16]:
# Initialize FAISS vector store
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    multi_process=True,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # set True for cosine similarity
)

knowledge_vector_database = FAISS.from_documents(
    docs_processed,
    embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  670MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [17]:
display(
RAW_KNOWLEDGE_BASE)

[Document(metadata={'channel_id': 2713, 'event_date': Timestamp('2024-04-30 23:55:00+0200', tz='UTC+02:00'), 'title': 'Peep And The Big Wide World', 'category': 'Adventure', 'synopsis': "'S1/E17 - The Fish Museum/Peep's Night Out'. Poor Fish Jr. She wants so much to wear sneakers and sit in a tree. Can Quack make her dreams come true? Goaded by Squeak, Peep decides to experience life after dark."}, page_content="Channel ID: 2713\nProgram Name: Peep And The Big Wide World\nProgram Category: Adventure\nProgram Synopsis: 'S1/E17 - The Fish Museum/Peep's Night Out'. Poor Fish Jr. She wants so much to wear sneakers and sit in a tree. Can Quack make her dreams come true? Goaded by Squeak, Peep decides to experience life after dark."),
 Document(metadata={'channel_id': 3627, 'event_date': Timestamp('2024-04-30 23:50:00+0200', tz='UTC+02:00'), 'title': 'Father Brown', 'category': 'Period Drama', 'synopsis': "'S5/E5 - The Hand of Lucia'. When notorious author Lucia Morrell is attacked, suspicio

In [18]:
import gradio as gr
# Function to retrieve and display results
def retrieve_results(query):
   results = knowledge_vector_database.similarity_search(query, k=5)
   return "\n\n".join(result.page_content for result in results)
# Gradio app
interface = gr.Interface(
   fn=retrieve_results,
   inputs=gr.Textbox(lines=2, placeholder="Type a message"),
   outputs=gr.Textbox(label="Recommendations", lines=10),
   title="TV Guide BOT",
   description="What would you like to watch?",
   theme="soft",
   examples=["Recommend Actions", "I want to watch cartoons", "Comedy Movies"],
   cache_examples=True,
)
interface.launch()

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Caching examples at: '/content/.gradio/cached_examples/16'
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a77c9a9277c3f96acc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
